In [1]:
import os
import cv2
import re
import easyocr
import numpy as np
import pandas as pd

from pathlib import Path
from ultralytics import YOLO
from datetime import datetime

In [2]:
vehicle_model = YOLO("models/vehicle_weights.pt")
helmet_model = YOLO("models/helmet_weights.pt")
triple_model = YOLO("models/triple_weights.pt")
plate_model = YOLO("models/license_weights.pt")

reader = easyocr.Reader(['en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


In [4]:
INPUT_FOLDER = "datasets/ocr_dataset"

EVIDENCE_FOLDER = "outputs/evidence"

CSV_FILE = "outputs/violations.csv"

os.makedirs(EVIDENCE_FOLDER, exist_ok=True)

In [5]:
if not os.path.exists(CSV_FILE):

    df = pd.DataFrame(columns=[
        "timestamp",
        "image_name",
        "vehicle_type",
        "plate_number",
        "helmet_violation",
        "triple_violation",
        "evidence_image"
    ])

    df.to_csv(CSV_FILE, index=False)

In [6]:
def preprocess_plate(plate_img):

    gray = cv2.cvtColor(
        plate_img,
        cv2.COLOR_BGR2GRAY
    )

    h, w = gray.shape

    gray = cv2.resize(
        gray,
        (w * 2, h * 2),
        interpolation=cv2.INTER_CUBIC
    )

    gray = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    thresh = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    return thresh

In [7]:
def clean_plate_number(text):

    text = text.upper()

    text = text.replace(" ", "")
    text = text.replace("-", "")

    text = re.sub(
        r'[^A-Z0-9]',
        '',
        text
    )

    replacements = {
        'O': '0',
        'I': '1',
        'S': '5',
        'B': '8'
    }

    cleaned = ""

    for ch in text:

        cleaned += replacements.get(
            ch,
            ch
        )

    return cleaned

In [8]:
def extract_plate_text(plate_crop):

    processed = preprocess_plate(
        plate_crop
    )

    results = reader.readtext(
        processed
    )

    best_text = "UNKNOWN"
    best_conf = 0

    for result in results:

        _, text, conf = result

        if conf > best_conf:

            best_conf = conf
            best_text = text

    best_text = clean_plate_number(
        best_text
    )

    if best_conf < 0.0:

        best_text = "UNKNOWN"

    return best_text, best_conf

In [9]:
def detect_motorcycles(image):

    motorcycles = []

    results = vehicle_model(
        image,
        verbose=False
    )

    for box in results[0].boxes:

        cls = int(box.cls[0])

        vehicle_name = vehicle_model.names[cls]

        if vehicle_name != "motorcycle":
            continue

        x1, y1, x2, y2 = map(
            int,
            box.xyxy[0]
        )

        crop = image[
            y1:y2,
            x1:x2
        ]

        motorcycles.append({
            "crop": crop,
            "bbox": (x1, y1, x2, y2)
        })

    return motorcycles

In [10]:
def check_helmet_violation(
    motorcycle_crop
):

    results = helmet_model(
        motorcycle_crop,
        verbose=False
    )

    violation = False

    for box in results[0].boxes:

        cls = int(box.cls[0])

        label = helmet_model.names[cls]

        if label == "Without Helmet":

            violation = True
            break

    return violation

In [11]:
def check_triple_riding(
    motorcycle_crop
):

    results = triple_model(
        motorcycle_crop,
        verbose=False
    )

    if len(results[0].boxes) == 0:
        return False

    best_box = max(
        results[0].boxes,
        key=lambda x: float(x.conf[0])
    )

    cls = int(best_box.cls[0])

    label = triple_model.names[cls]

    return label == "triple_rider"

In [12]:
def detect_license_plate(
    motorcycle_crop
):

    results = plate_model(
        motorcycle_crop,
        verbose=False
    )

    if len(results[0].boxes) == 0:
        return None, None

    largest_area = 0

    best_crop = None

    best_bbox = None

    for box in results[0].boxes:

        x1, y1, x2, y2 = map(
            int,
            box.xyxy[0]
        )

        area = (
            x2 - x1
        ) * (
            y2 - y1
        )

        if area > largest_area:

            largest_area = area

            best_crop = motorcycle_crop[
                y1:y2,
                x1:x2
            ]

            best_bbox = (
                x1,
                y1,
                x2,
                y2
            )

    return best_crop, best_bbox

In [13]:
def append_to_csv(row):

    columns = [
        "timestamp",
        "image_name",
        "vehicle_type",
        "plate_number",
        "helmet_violation",
        "triple_violation",
        "evidence_image"
    ]

    try:
        df = pd.read_csv(
            CSV_FILE
        )

    except (
        FileNotFoundError,
        pd.errors.EmptyDataError
    ):
        df = pd.DataFrame(
            columns=columns
        )

    df.loc[len(df)] = row

    df.to_csv(
        CSV_FILE,
        index=False
    )

In [14]:
def save_evidence(
    image,
    vehicle_bbox,
    plate_number,
    helmet_violation,
    triple_violation,
    image_name
):

    x1, y1, x2, y2 = vehicle_bbox

    evidence = image.copy()

    cv2.rectangle(
        evidence,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

    violations = []

    if helmet_violation:
        violations.append("helmet")

    if triple_violation:
        violations.append("triple")

    violation_text = "_".join(violations)

    label = (
        f"Plate:{plate_number} | "
        f"{violation_text}"
    )

    cv2.putText(
        evidence,
        label,
        (x1, max(30, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 0, 255),
        2
    )

    base_name = Path(image_name).stem
    extension = Path(image_name).suffix

    evidence_name = (
        f"{base_name}_{violation_text}{extension}"
    )

    evidence_image = os.path.join(
        EVIDENCE_FOLDER,
        evidence_name
    )

    cv2.imwrite(
        evidence_image,
        evidence
    )

    return evidence_image

In [15]:
def process_image(image_path):

    image = cv2.imread(
        image_path
    )

    motorcycles = detect_motorcycles(
        image
    )

    for vehicle in motorcycles:

        crop = vehicle["crop"]

        bbox = vehicle["bbox"]

        helmet_violation = (
            check_helmet_violation(
                image
            )
        )

        triple_violation = (
            check_triple_riding(
                crop
            )
        )

        violation_found = (
            helmet_violation
            or
            triple_violation
        )

        if not violation_found:
            continue

        plate_crop, plate_bbox = (
            detect_license_plate(
                image
            )
        )

        if plate_crop is None:

            plate_number = "UNKNOWN"

        else:

            plate_number, conf = (
                extract_plate_text(
                    plate_crop
                )
            )

        evidence_image = save_evidence(
            image=image,
            vehicle_bbox=bbox,
            plate_number=plate_number,
            helmet_violation=helmet_violation,
            triple_violation=triple_violation,
            image_name=os.path.basename(
                image_path
            )
        )

        append_to_csv([
            datetime.now(),
            os.path.basename(
                image_path
            ),
            "motorcycle",
            plate_number,
            helmet_violation,
            triple_violation,
            evidence_image
        ])

In [19]:
image_extensions = (
    "*.jpg",
    "*.jpeg",
    "*.png"
)

for ext in image_extensions:

    for image_path in Path(
        INPUT_FOLDER
    ).glob(ext):

        print(
            f"Processing : {image_path.name}"
        )

        process_image(
            str(image_path)
        )

print(
    "\nPipeline Completed Successfully"
)

Processing : no_vioilation.jpg
Processing : helmet_1.png
Processing : triple_1.png
Processing : violation_1.png

Pipeline Completed Successfully


In [17]:
# os.remove(CSV_FILE)